# Training a Conditional StyleGAN2 Model using PyTorch / ONNX

This notebook trains an image-to-image generator with the same data as the pix2pix notebooks
(1024×512 pairs, target on the left, input on the right) but with a **StyleGAN2** synthesis
network instead of a U-Net: modulated convolutions, a residual discriminator with R1, an EMA
generator. The latent `z` of StyleGAN is kept and joined by a vector from the input image, so
`w = mapping(z, input)` styles every layer, while an encoder feeds the input image into the
synthesis network at every resolution.

**Why this one:** on a 15k-pair face-mesh → photo dataset it beat pix2pix on every quality
number after 4 epochs, and the configuration set as default here ("V8": half-width channel
plan, additive skips, a small encoder) runs at 26 fps in Figment on an M2 Max, against 18 fps
for the pix2pix U-Net and 6 fps for the full-width StyleGAN. Set `channel_base = 32768`,
`skip_mode = "concat"`, `encoder_scale = 1.0`, `encoder_top_conv = True` for the full-width
version if speed does not matter.

**Defaults:** 512×512 per half, batch 16, about 12 GB of GPU memory. The generator,
discriminator and VGG loss run through `torch.compile`: 38 images/s on an RTX 4090 (7 min per
epoch of 15k pairs). The same run without compiling is 23 images/s. Compiling takes about a
minute at the first iteration; set `compile_models = False` on a GPU or PyTorch build where it
fails.

**Output:** `generator_epoch_XXX.onnx` (fp32) and `generator_epoch_XXX_fp16.onnx` (fp16
weights, fp32 in and out) every `snapshot_interval` epochs. Both take one `input`
`[1, 3, H, W]` in `[-1, 1]` and return one `output` of the same shape, exactly like the
pix2pix exports, so they load in Figment's ONNX Image Model node. The fp16 file is the one to
use there: same picture, a third of the size, faster. Only the `keep_snapshots` most recent
`.pth` files are kept (824 MB each); the ONNX files stay.

**Validation:** `val_pairs` pairs, evenly spaced through the dataset, are held out of training.
The progress grid shows four of them, and every `sample_interval` iterations the log gets the
EMA generator's L1 and VGG distance on all of them (`val_l1`, `val_vgg`). Training L1 on pairs
the network has seen keeps falling long after the held-out numbers have stopped; the held-out
VGG is the number to watch to decide when to stop.

In [ ]:
# Make sure you are connected to a runtime with a GPU
!nvidia-smi -L

In [ ]:
# Install required packages
!pip install -q matplotlib tqdm onnx onnxruntime onnxconverter-common

In [ ]:
# Import all dependencies
import copy
import glob
import math
import os
import random
import sys
import time
import warnings
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.onnx
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
import torchvision.transforms.functional as TF
from torchvision.utils import save_image

from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [ ]:
# Check if GPU is available
gpu_available = torch.cuda.is_available()
print("GPU is", "available" if gpu_available else "NOT AVAILABLE")

In [ ]:
# OPTIONAL: If you don't have a dataset yet, you can download a pre-existing one.
# If you have a dataset already, you can skip this step.
#!curl -o ds.zip https://algorithmicgaze.s3.amazonaws.com/workshops/2025-research-week/prep/2025-10-03-dataset-pose-points.zip
#!mkdir -p datasets/faces
#!unzip -j -o -qq *.zip -d datasets/faces
# Remove macOS metadata cruft
#!rm -rf datasets/faces/._*

In [ ]:
# Some helper functions for creating/checking directories.
def directory_should_exist(*args):
    dir = os.path.join(*args)
    if not os.path.isdir(dir):
        raise Exception("Path '{}' is not a directory.".format(dir))
    return dir

def ensure_directory(*args):
    dir = os.path.join(*args)
    os.makedirs(dir, exist_ok=True)
    return dir

In [ ]:
# Point the script to the correct dataset folder and configure training.
input_dir = directory_should_exist("datasets/faces")
output_dir = ensure_directory("output-stylegan2-conditional")
log_file_path = os.path.join(output_dir, "training_log.txt")

# --- Image settings ---
image_size = 512           # each half of the pair is resized to this; must be a multiple of 128
jitter_size = 60           # pairs are enlarged by this many pixels and randomly cropped back

# --- Architecture (defaults = the fast "V8" configuration) ---
channel_base = 16384       # generator channels = min(channel_max, channel_base // resolution); 32768 = full width
channel_max = 512
d_channel_base = 32768     # the discriminator keeps its full width whatever the generator does
skip_mode = "add"          # how encoder features join the synthesis: "add" (thin) or "concat" (U-Net style, 2x the cost)
encoder_scale = 0.5        # encoder channels relative to the generator's; the input is thin lines, it needs little
encoder_top_conv = False   # keep the encoder's 3x3 conv at full resolution (costs as much as a synthesis conv)
z_dim = 512                # latent
w_dim = 512                # style vector
c_dim = 512                # conditioning vector pooled from the encoder
mapping_layers = 4
compile_models = True      # torch.compile G, D and the VGG loss: 1.65x faster, ~1 min warm-up at the first iteration

# --- Training ---
epochs = 100
batch_size = 16            # a multiple of 4 (minibatch std group size)
lr = 2e-3
lambda_l1 = 10.0           # pixel loss against the target
lambda_vgg = 10.0          # VGG19 perceptual loss against the target
r1_every = 16              # lazy R1 gradient penalty interval
r1_gamma = 10.0
ema_kimg = 10.0            # EMA half-life in thousands of images (ramps up early on)
seed = 0

# --- Logging ---
sample_interval = 250      # iterations between sample images (counted over the whole run, not per epoch)
snapshot_interval = 2      # epochs between .pth + .onnx snapshots
keep_snapshots = 3         # .pth files to keep; older ones are deleted, ONNX exports are kept
val_pairs = 8              # pairs held out of training for the progress grid and the val_l1 / val_vgg numbers
num_workers = 6

torch.manual_seed(seed)
random.seed(seed)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
# Create the dataset class. Left half = target, right half = input (the conditioning image).
# Same augmentation as the pix2pix notebooks: enlarge, random crop, random horizontal flip.

class PairDataset(Dataset):
    def __init__(self, root_dir, image_size=512, jitter_size=60, augment=False, transform=None):
        self.root_dir = root_dir
        self.image_size = image_size
        self.jitter_size = jitter_size
        self.augment = augment
        self.transform = transform
        self.image_files = sorted(
            f for f in os.listdir(root_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image = Image.open(os.path.join(self.root_dir, self.image_files[idx]))
        if image.mode != "RGB":
            image = image.convert("RGB")
        w, h = image.size
        target = image.crop((0, 0, w // 2, h))
        cond = image.crop((w // 2, 0, w, h))

        size = self.image_size
        if self.augment:
            big = [size + self.jitter_size, size + self.jitter_size]
            cond = TF.resize(cond, big, interpolation=TF.InterpolationMode.BICUBIC)
            target = TF.resize(target, big, interpolation=TF.InterpolationMode.BICUBIC)
            i, j, ch, cw = transforms.RandomCrop.get_params(cond, output_size=(size, size))
            cond, target = TF.crop(cond, i, j, ch, cw), TF.crop(target, i, j, ch, cw)
            if random.random() > 0.5:
                cond, target = TF.hflip(cond), TF.hflip(target)
        else:
            cond = TF.resize(cond, [size, size], interpolation=TF.InterpolationMode.BICUBIC)
            target = TF.resize(target, [size, size], interpolation=TF.InterpolationMode.BICUBIC)

        if self.transform:
            cond = self.transform(cond)
            target = self.transform(target)
        return cond, target

In [ ]:
# Normalize to [-1, 1] so the generator's output matches.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

full_dataset = PairDataset(input_dir, image_size=image_size, jitter_size=jitter_size, transform=transform)

# Held-out pairs, evenly spaced through the (sorted) file list: never trained on. Four of them
# make the progress grid; all of them give the val_l1 / val_vgg numbers in the log.
val_idx = [int(i) for i in np.linspace(0, len(full_dataset) - 1, val_pairs)] if val_pairs > 0 else []
train_idx = [i for i in range(len(full_dataset)) if i not in set(val_idx)]
dataset = Subset(full_dataset, train_idx)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                        pin_memory=True, drop_last=True, persistent_workers=True)

fixed_dataset = PairDataset(input_dir, image_size=image_size, augment=False, transform=transform)
grid_idx = val_idx[:4] if val_idx else [int(i) for i in np.linspace(0, len(fixed_dataset) - 1, 4)]
fixed_cond = torch.stack([fixed_dataset[i][0] for i in grid_idx])
fixed_target = torch.stack([fixed_dataset[i][1] for i in grid_idx])
val_cond = torch.stack([fixed_dataset[i][0] for i in val_idx]) if val_idx else None
val_target = torch.stack([fixed_dataset[i][1] for i in val_idx]) if val_idx else None

print(f"Dataset: {len(dataset)} training pairs + {len(val_idx)} held out, at {image_size}x{image_size}")

In [ ]:
# Show a single image pair from the dataset
def plot_tensor(ax, title, img):
    img = (img + 1) / 2
    img = img.clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

cond_img, target_img = next(iter(dataloader))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
plot_tensor(ax1, "Input Image", cond_img[0])
plot_tensor(ax2, "Target Image", target_img[0])
plt.show()

In [ ]:
# === Equalized learning rate ===
# Weights are initialized ~ N(0, 1) and rescaled by 1/sqrt(fan_in) at forward time, so every
# layer trains at the same effective rate. LeakyReLU is scaled by sqrt(2) to keep the variance.

def lrelu(x):
    return F.leaky_relu(x, 0.2) * math.sqrt(2)


def normalize_2nd_moment(x, eps=1e-8):
    return x * torch.rsqrt(x.square().mean(dim=1, keepdim=True) + eps)


class EqualLinear(nn.Module):
    def __init__(self, fin, fout, bias=True, bias_init=0.0, lr_mul=1.0, act=False):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(fout, fin) / lr_mul)
        self.bias = nn.Parameter(torch.full([fout], float(bias_init))) if bias else None
        self.scale = lr_mul / math.sqrt(fin)
        self.lr_mul = lr_mul
        self.act = act

    def forward(self, x):
        b = self.bias * self.lr_mul if self.bias is not None else None
        x = F.linear(x, self.weight * self.scale, b)
        return lrelu(x) if self.act else x


class EqualConv(nn.Module):
    def __init__(self, cin, cout, k, bias=True, act=True, stride=1):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(cout, cin, k, k))
        self.bias = nn.Parameter(torch.zeros(cout)) if bias else None
        self.scale = 1 / math.sqrt(cin * k * k)
        self.pad = k // 2
        self.stride = stride
        self.act = act

    def forward(self, x):
        x = F.conv2d(x, self.weight * self.scale, self.bias, stride=self.stride, padding=self.pad)
        return lrelu(x) if self.act else x

In [ ]:
# === Modulated convolution (the core StyleGAN2 layer) ===
# The style scales each input channel, a plain conv runs, and the output channels are scaled
# back by the demodulation factor. Written this way the conv weights stay a static tensor and
# the graph exports to ONNX as Conv, Mul and MatMul: everything runs on the GPU in
# onnxruntime-web, and there is no InstanceNormalization to overflow in fp16.

class ModConv(nn.Module):
    def __init__(self, cin, cout, k, w_dim, demod=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(cout, cin, k, k))
        self.affine = EqualLinear(w_dim, cin, bias_init=1.0)
        self.scale = 1 / math.sqrt(cin * k * k)
        self.pad = k // 2
        self.demod = demod

    def forward(self, x, w):
        s = self.affine(w)                                       # [B, cin]
        weight = self.weight * self.scale
        x = x * s.to(x.dtype)[:, :, None, None]
        x = F.conv2d(x, weight.to(x.dtype), padding=self.pad)
        if self.demod:
            w2 = weight.float().square().sum(dim=[2, 3])         # [cout, cin]
            d = torch.rsqrt(s.float().square() @ w2.t() + 1e-8)  # [B, cout]
            x = x * d.to(x.dtype)[:, :, None, None]
        return x


class StyledConv(nn.Module):
    """Modulated 3x3 conv + noise + bias + lrelu. The noise buffer is fixed at export time."""

    def __init__(self, cin, cout, w_dim, size):
        super().__init__()
        self.conv = ModConv(cin, cout, 3, w_dim)
        self.noise_strength = nn.Parameter(torch.zeros([]))
        self.bias = nn.Parameter(torch.zeros(cout))
        self.register_buffer("noise_const", torch.randn(1, 1, size, size))

    def forward(self, x, w, noise_mode):
        x = self.conv(x, w)
        if noise_mode == "random":
            noise = torch.randn(x.shape[0], 1, x.shape[2], x.shape[3], device=x.device, dtype=x.dtype)
            x = x + noise * self.noise_strength.to(x.dtype)
        elif noise_mode == "const":
            x = x + self.noise_const.to(x.dtype) * self.noise_strength.to(x.dtype)
        return lrelu(x + self.bias.to(x.dtype)[None, :, None, None])


class ToRGB(nn.Module):
    def __init__(self, cin, w_dim):
        super().__init__()
        self.conv = ModConv(cin, 3, 1, w_dim, demod=False)
        self.bias = nn.Parameter(torch.zeros(3))

    def forward(self, x, w):
        return self.conv(x, w) + self.bias.to(x.dtype)[None, :, None, None]


def upsample(x):
    return F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)


def channel_plan(num_levels, base, cmax, top_res):
    """Channels per level, level 0 = the smallest. StyleGAN2 rule: base / resolution."""
    return [min(base // (top_res >> (num_levels - 1 - i)), cmax) for i in range(num_levels)]

In [ ]:
# === Encoder: the conditioning image ===
# One 3x3 conv per resolution on the way down, strided convs between levels. It returns a
# feature map per level (the skips into the synthesis network) and a vector pooled from the
# smallest level (the conditioning half of the mapping network's input).

class Encoder(nn.Module):
    def __init__(self, chans, base_size, c_dim, top_conv=True):
        super().__init__()
        self.from_rgb = EqualConv(3, chans[-1], 1)
        self.convs = nn.ModuleList()
        self.downs = nn.ModuleList()
        for i in range(len(chans) - 1, 0, -1):
            keep = top_conv or i < len(chans) - 1
            self.convs.append(EqualConv(chans[i], chans[i], 3) if keep else nn.Identity())
            self.downs.append(EqualConv(chans[i], chans[i - 1], 3, stride=2))
        self.conv0 = EqualConv(chans[0], chans[0], 3)
        self.fc = EqualLinear(chans[0] * base_size * base_size, c_dim, act=True)

    def forward(self, cond):
        x = self.from_rgb(cond)
        feats = []
        for conv, down in zip(self.convs, self.downs):
            x = conv(x)
            feats.append(x)
            x = down(x)
        x = self.conv0(x)
        feats.append(x)
        feats.reverse()  # level 0 first
        return feats, self.fc(x.flatten(1))

In [ ]:
# === Mapping network (z, cond) -> w ===
# StyleGAN maps a random z to the style w. Here the conditioning vector from the encoder is
# embedded, normalized and concatenated with z first, so w also depends on the input image:
# pose and expression modulate the filters, not only the spatial skips.

class Mapping(nn.Module):
    def __init__(self, z_dim, c_dim, w_dim, num_layers=4, lr_mul=0.01):
        super().__init__()
        self.embed = EqualLinear(c_dim, w_dim)
        layers = []
        fin = z_dim + w_dim
        for _ in range(num_layers):
            layers.append(EqualLinear(fin, w_dim, lr_mul=lr_mul, act=True))
            fin = w_dim
        self.layers = nn.ModuleList(layers)

    def forward(self, z, c):
        x = torch.cat([normalize_2nd_moment(z), normalize_2nd_moment(self.embed(c))], dim=1)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
# === Synthesis ===
# StyleGAN2's skip architecture: each level upsamples, joins the encoder feature of that
# level, runs two styled convs and adds a ToRGB to the upsampled RGB so far. The encoder's
# smallest feature map replaces StyleGAN's learned constant input.
#
# skip_mode "concat" stacks the upsampled features with the encoder feature (a U-Net); the
# first conv then reads a bundle 3x as wide. "add" projects both to the level's width and
# sums them, which halves the cost at the same quality on our data.

def project(cin, cout):
    """1x1 linear projection, or nothing when the widths already match."""
    return EqualConv(cin, cout, 1, bias=False, act=False) if cin != cout else nn.Identity()


class Synthesis(nn.Module):
    def __init__(self, chans, ech, sizes, w_dim, skip="add"):
        super().__init__()
        self.skip = skip
        self.conv_in = StyledConv(ech[0], chans[0], w_dim, sizes[0])
        self.rgb_in = ToRGB(chans[0], w_dim)
        self.conv0 = nn.ModuleList()
        self.conv1 = nn.ModuleList()
        self.to_rgb = nn.ModuleList()
        self.up_proj = nn.ModuleList()
        self.skip_proj = nn.ModuleList()
        for i in range(1, len(chans)):
            c_prev, c, e = chans[i - 1], chans[i], ech[i]
            if skip == "concat":
                self.up_proj.append(nn.Identity())
                self.skip_proj.append(nn.Identity())
                cin0 = c_prev + e
            else:
                self.up_proj.append(project(c_prev, c))
                self.skip_proj.append(project(e, c))
                cin0 = c
            self.conv0.append(StyledConv(cin0, c, w_dim, sizes[i]))
            self.conv1.append(StyledConv(c, c, w_dim, sizes[i]))
            self.to_rgb.append(ToRGB(c, w_dim))

    def forward(self, feats, w, noise_mode):
        x = self.conv_in(feats[0], w, noise_mode)
        rgb = self.rgb_in(x, w)
        for i in range(1, len(feats)):
            up = upsample(x)
            e = self.skip_proj[i - 1](feats[i])
            x = torch.cat([up, e], dim=1) if self.skip == "concat" else self.up_proj[i - 1](up) + e
            x = self.conv0[i - 1](x, w, noise_mode)
            x = self.conv1[i - 1](x, w, noise_mode)
            rgb = upsample(rgb) + self.to_rgb[i - 1](x, w)
        return rgb

In [ ]:
# === Generator ===
# Encoder -> (features, c); mapping(z, c) -> w; synthesis(features, w) -> image.
# ExportGenerator bakes a fixed z and the fixed noise buffers into the graph and clamps the
# output, so the ONNX has one image input and one image output, like the pix2pix exports.

class Generator(nn.Module):
    def __init__(self, image_size, z_dim=512, w_dim=512, c_dim=512, channel_base=16384,
                 channel_max=512, mapping_layers=4, enc_scale=0.5, enc_top_conv=False,
                 skip="add", num_levels=8):
        super().__init__()
        assert image_size % (1 << (num_levels - 1)) == 0, "image_size must be a multiple of 128"
        self.z_dim = z_dim
        sizes = [image_size >> (num_levels - 1 - i) for i in range(num_levels)]
        chans = channel_plan(num_levels, channel_base, channel_max, image_size)
        ech = [max(8, int(c * enc_scale)) for c in chans]
        self.encoder = Encoder(ech, sizes[0], c_dim, enc_top_conv)
        self.mapping = Mapping(z_dim, c_dim, w_dim, mapping_layers)
        self.synthesis = Synthesis(chans, ech, sizes, w_dim, skip)
        self.chans, self.ech = chans, ech

    def forward(self, cond, z, noise_mode="random"):
        feats, c = self.encoder(cond)
        if z.shape[0] != c.shape[0]:
            z = z.repeat(c.shape[0], 1)
        w = self.mapping(z, c)
        return self.synthesis(feats, w, noise_mode)


class ExportGenerator(nn.Module):
    def __init__(self, generator, z):
        super().__init__()
        self.generator = generator
        self.register_buffer("z", z)

    def forward(self, cond):
        return self.generator(cond, self.z, noise_mode="const").clamp(-1, 1)

In [ ]:
# === Discriminator ===
# StyleGAN2's residual discriminator on the pair [input | image] (6 channels), so it judges
# whether the image fits the input, like pix2pix's conditional PatchGAN. MinibatchStd at the
# smallest level adds a channel with the std across a group of 4 samples.

class MinibatchStd(nn.Module):
    def __init__(self, group=4):
        super().__init__()
        self.group = group

    def forward(self, x):
        n, c, h, w = x.shape
        g = min(self.group, n)
        while n % g != 0:
            g -= 1
        y = x.float().reshape(g, -1, c, h, w)
        y = y - y.mean(dim=0)
        y = y.square().mean(dim=0)
        y = (y + 1e-8).sqrt().mean(dim=[1, 2, 3])
        y = y.reshape(-1, 1, 1, 1).repeat(g, 1, h, w)
        return torch.cat([x, y.to(x.dtype)], dim=1)


class DBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv0 = EqualConv(cin, cin, 3)
        self.conv1 = EqualConv(cin, cout, 3, stride=2)
        self.skip = EqualConv(cin, cout, 1, bias=False, act=False)

    def forward(self, x):
        y = self.skip(F.avg_pool2d(x, 2))
        x = self.conv1(self.conv0(x))
        return (x + y) / math.sqrt(2)


class Discriminator(nn.Module):
    def __init__(self, image_size, channel_base=32768, channel_max=512, num_levels=8):
        super().__init__()
        chans = channel_plan(num_levels, channel_base, channel_max, image_size)
        self.from_rgb = EqualConv(6, chans[-1], 1)
        self.blocks = nn.ModuleList(DBlock(chans[i], chans[i - 1]) for i in range(num_levels - 1, 0, -1))
        self.mbstd = MinibatchStd()
        self.conv = EqualConv(chans[0] + 1, chans[0], 3)
        base = image_size >> (num_levels - 1)
        self.fc = EqualLinear(chans[0] * base * base, chans[0], act=True)
        self.out = EqualLinear(chans[0], 1)

    def forward(self, cond, image):
        x = self.from_rgb(torch.cat([cond, image], dim=1))
        for block in self.blocks:
            x = block(x)
        x = self.conv(self.mbstd(x))
        return self.out(self.fc(x.flatten(1)))

In [ ]:
# === Losses and regularization ===
# Generator: non-saturating logistic + L1 + VGG19 perceptual against the target (pix2pixHD's
# recipe, with a lower L1 weight so the GAN term owns the texture).
# Discriminator: non-saturating logistic + lazy R1.

class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features.eval()
        for p in vgg.parameters():
            p.requires_grad_(False)
        bounds = [0, 2, 7, 12, 21, 30]  # relu1_1, relu2_1, relu3_1, relu4_1, relu5_1
        self.slices = nn.ModuleList(nn.Sequential(*vgg[bounds[i]:bounds[i + 1]]) for i in range(5))
        self.weights = [1 / 32, 1 / 16, 1 / 8, 1 / 4, 1.0]
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x, y):
        x = ((x + 1) / 2 - self.mean) / self.std
        y = ((y + 1) / 2 - self.mean) / self.std
        loss = 0.0
        for w, s in zip(self.weights, self.slices):
            x, y = s(x), s(y)
            loss = loss + w * F.l1_loss(x, y.detach())
        return loss


def d_logistic_loss(d_real, d_fake):
    return F.softplus(-d_real).mean() + F.softplus(d_fake).mean()


def g_nonsaturating_loss(d_fake):
    return F.softplus(-d_fake).mean()


def r1_penalty(discriminator, cond, real):
    real = real.detach().requires_grad_(True)
    d_real = discriminator(cond, real)
    grad = torch.autograd.grad(d_real.float().sum(), real, create_graph=True)[0]
    return grad.float().square().sum(dim=[1, 2, 3]).mean()

In [ ]:
# === Exponential moving average of generator weights ===
# The EMA generator makes every sample image and every export. Its half-life ramps up with
# the number of images seen, so early training is not averaged with random weights.

def requires_grad(module, flag):
    for p in module.parameters():
        p.requires_grad_(flag)


@torch.no_grad()
def update_ema(g_ema, generator, cur_nimg, batch_size, ema_kimg):
    ema_nimg = min(ema_kimg * 1000, cur_nimg * 0.05)
    beta = 0.5 ** (batch_size / max(ema_nimg, 1e-8))
    for p_ema, p in zip(g_ema.parameters(), generator.parameters()):
        p_ema.lerp_(p, 1 - beta)
    for b_ema, b in zip(g_ema.buffers(), generator.buffers()):
        b_ema.copy_(b)

In [ ]:
# === Snapshot save/load + ONNX export ===

def get_latest_snapshot(output_dir):
    snapshots = glob.glob(os.path.join(output_dir, "snapshot_epoch_*.pth"))
    if not snapshots:
        return None
    return max(snapshots, key=os.path.getctime)


def save_snapshot(path, epoch, step, generator, g_ema, discriminator, g_optimizer, d_optimizer):
    torch.save({
        "epoch": epoch,
        "step": step,
        "generator": generator.state_dict(),
        "g_ema": g_ema.state_dict(),
        "discriminator": discriminator.state_dict(),
        "g_optimizer": g_optimizer.state_dict(),
        "d_optimizer": d_optimizer.state_dict(),
    }, path)


def load_snapshot(path, generator, g_ema, discriminator, g_optimizer, d_optimizer, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    generator.load_state_dict(ckpt["generator"])
    g_ema.load_state_dict(ckpt["g_ema"])
    discriminator.load_state_dict(ckpt["discriminator"])
    g_optimizer.load_state_dict(ckpt["g_optimizer"])
    d_optimizer.load_state_dict(ckpt["d_optimizer"])
    return ckpt["epoch"], ckpt["step"]


def export_onnx(g_ema, path, image_size, z, device):
    """Static shape, fp32, opset 17: one `input` [1, 3, H, W] in [-1, 1], one `output`."""
    model = ExportGenerator(copy.deepcopy(g_ema).float().eval(), z).to(device).eval()
    dummy = torch.randn(1, 3, image_size, image_size, device=device)
    with torch.no_grad():
        model(dummy)  # warm up
    torch.onnx.export(
        model,
        dummy,
        path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamo=False,
    )


def export_fp16(src, dst):
    """fp16 weights and compute with fp32 input/output, the file to load in Figment.
    Drops the no-op Cast nodes, folds constants while still fp32, keeps Resize scales fp32."""
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, numpy_helper
    from onnxconverter_common import float16

    model = onnx.load(src)
    graph = model.graph
    rename, kept = {}, []
    for node in graph.node:
        if node.op_type == "Cast" and node.attribute[0].i == TensorProto.FLOAT:
            rename[node.output[0]] = node.input[0]
        else:
            kept.append(node)
    def resolve(name):
        while name in rename:
            name = rename[name]
        return name
    for node in kept:
        for i, name in enumerate(node.input):
            node.input[i] = resolve(name)
    for out in graph.output:
        out.name = resolve(out.name)
    del graph.node[:]
    graph.node.extend(kept)
    del graph.value_info[:]

    folded = dst + ".folded.onnx"
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
    so.optimized_model_filepath = folded
    ort.InferenceSession(model.SerializeToString(), so, providers=["CPUExecutionProvider"])
    model = onnx.load(folded)
    os.remove(folded)

    block = [op for op in float16.DEFAULT_OP_BLOCK_LIST if op not in ("Resize", "Upsample")]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model16 = float16.convert_float_to_float16(model, keep_io_types=True, op_block_list=block)
    producers = {n.output[0]: n for n in model16.graph.node if n.op_type == "Constant"}
    inits = {i.name: i for i in model16.graph.initializer}
    for node in model16.graph.node:
        if node.op_type != "Resize":
            continue
        for name in node.input[1:3]:
            if name in producers and producers[name].attribute[0].t.data_type == TensorProto.FLOAT16:
                t = producers[name].attribute[0].t
                t.CopyFrom(numpy_helper.from_array(numpy_helper.to_array(t).astype(np.float32), name))
            elif name in inits and inits[name].data_type == TensorProto.FLOAT16:
                inits[name].CopyFrom(numpy_helper.from_array(numpy_helper.to_array(inits[name]).astype(np.float32), name))
    onnx.save(model16, dst)

In [ ]:
# === Training loop ===

def train(generator, discriminator, dataloader, opts):
    g_optimizer = optim.Adam(generator.parameters(), lr=opts.lr, betas=(0.0, 0.99), eps=1e-8)
    d_optimizer = optim.Adam(discriminator.parameters(), lr=opts.lr, betas=(0.0, 0.99), eps=1e-8)

    # EMA generator: used for sample images + ONNX export
    g_ema = copy.deepcopy(generator).eval()
    requires_grad(g_ema, False)

    vgg = VGGLoss().to(device) if opts.lambda_vgg > 0 else None
    export_z = torch.randn(1, opts.z_dim, generator=torch.Generator().manual_seed(opts.seed)).to(device)
    fixed_cond, fixed_target = opts.fixed_cond.to(device), opts.fixed_target.to(device)
    autocast = lambda: torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.type == "cuda")

    # Compiled copies share the parameters with the eager modules. The R1 penalty keeps using the
    # eager discriminator: it needs a double backward, which torch.compile does not support.
    g_run, d_run, vgg_eval = generator, discriminator, vgg
    if opts.compile_models and device.type == "cuda":
        g_run, d_run = torch.compile(generator), torch.compile(discriminator)
        if vgg is not None:
            vgg = torch.compile(vgg)

    @torch.no_grad()
    def validate():
        """EMA generator on the held-out pairs: L1 and (uncompiled) VGG distance to the targets."""
        if opts.val_cond is None or vgg_eval is None:
            return float("nan"), float("nan")
        l1s, vggs = [], []
        for k in range(0, len(opts.val_cond), 8):
            c, t = opts.val_cond[k:k + 8].to(device), opts.val_target[k:k + 8].to(device)
            with autocast():
                out = g_ema(c, export_z, noise_mode="const").float().clamp(-1, 1)
            l1s.append(F.l1_loss(out, t).item())
            vggs.append(vgg_eval(out, t).item())
        return float(np.mean(l1s)), float(np.mean(vggs))

    start_epoch, step = 1, 0
    if not opts.restart:
        latest = get_latest_snapshot(opts.output_dir)
        if latest:
            last_epoch, step = load_snapshot(latest, generator, g_ema, discriminator, g_optimizer, d_optimizer, device)
            start_epoch = last_epoch + 1
            print(f"Resuming from epoch {last_epoch}")
        else:
            print("No snapshots found. Starting from scratch.")
    else:
        print("Restarting from scratch.")

    for epoch in range(start_epoch, opts.epochs + 1):
        generator.train()
        discriminator.train()
        for i, (cond, real) in enumerate(tqdm(dataloader, file=sys.stdout)):
            cond, real = cond.to(device, non_blocking=True), real.to(device, non_blocking=True)
            B = cond.shape[0]

            # ---------------- Discriminator ----------------
            requires_grad(discriminator, True)
            requires_grad(generator, False)
            z = torch.randn(B, opts.z_dim, device=device)
            with autocast():
                with torch.no_grad():
                    fake = g_run(cond, z)
                d_fake = d_run(cond, fake)
                d_real = d_run(cond, real)
                d_loss = d_logistic_loss(d_real, d_fake)
            d_optimizer.zero_grad(set_to_none=True)
            d_loss.backward()

            # Lazy R1: every r1_every steps, scaled by r1_every to match the continuous version
            r1_val = 0.0
            if opts.r1_gamma > 0 and step % opts.r1_every == 0:
                with autocast():
                    r1 = r1_penalty(discriminator, cond, real)
                ((opts.r1_gamma / 2) * r1 * opts.r1_every).backward()
                r1_val = float(r1.item())
            d_optimizer.step()

            # ---------------- Generator ----------------
            requires_grad(discriminator, False)
            requires_grad(generator, True)
            z = torch.randn(B, opts.z_dim, device=device)
            with autocast():
                fake = g_run(cond, z)
                g_adv = g_nonsaturating_loss(d_run(cond, fake))
                g_l1 = F.l1_loss(fake.float(), real)
                g_vgg = vgg(fake.float(), real) if vgg is not None else torch.zeros((), device=device)
                g_loss = g_adv + opts.lambda_l1 * g_l1 + opts.lambda_vgg * g_vgg
            g_optimizer.zero_grad(set_to_none=True)
            g_loss.backward()
            g_optimizer.step()

            step += 1
            update_ema(g_ema, generator, step * B, B, opts.ema_kimg)

            # ---------------- Logging ----------------
            if i % 10 == 0:
                log_msg = (
                    f"Epoch {epoch} iter {i} | "
                    f"d_loss: {d_loss.item():.3f} | "
                    f"g_loss: {g_loss.item():.3f} | "
                    f"g_adv: {g_adv.item():.3f} | "
                    f"l1: {g_l1.item():.4f} | "
                    f"vgg: {g_vgg.item():.4f} | "
                    f"r1: {r1_val:.3f}"
                )
                tqdm.write(log_msg)
                with open(opts.log_file_path, "a") as f:
                    f.write(log_msg + "\n")

            if step % opts.sample_interval == 0:
                val_l1, val_vgg = validate()
                val_msg = f"Epoch {epoch} iter {i} | val_l1: {val_l1:.4f} | val_vgg: {val_vgg:.4f}"
                tqdm.write(val_msg)
                with open(opts.log_file_path, "a") as f:
                    f.write(val_msg + "\n")
                with torch.no_grad(), autocast():
                    samples = g_ema(fixed_cond, export_z, noise_mode="const").float().clamp(-1, 1)
                rows = torch.cat([fixed_cond, samples, fixed_target], dim=3).cpu()  # [input | output | target]
                grid = (rows + 1) / 2
                clear_output(wait=True)
                print(f"Epoch {epoch} | val_l1 {val_l1:.4f} | val_vgg {val_vgg:.4f}")
                fig, axes = plt.subplots(len(grid), 1, figsize=(12, 4 * len(grid)))
                for ax, row in zip(axes, grid):
                    ax.imshow(row.permute(1, 2, 0).numpy())
                    ax.axis("off")
                plt.show()
                save_image(grid, f"{opts.output_dir}/epoch_{epoch}_iter_{i}.jpg", nrow=1, padding=0)

        if epoch % opts.snapshot_interval == 0:
            snapshot_path = f"{opts.output_dir}/snapshot_epoch_{epoch}.pth"
            save_snapshot(snapshot_path, epoch, step, generator, g_ema, discriminator, g_optimizer, d_optimizer)
            onnx_path = f"{opts.output_dir}/generator_epoch_{epoch}.onnx"
            export_onnx(g_ema, onnx_path, opts.image_size, export_z, device)
            fp16_path = f"{opts.output_dir}/generator_epoch_{epoch}_fp16.onnx"
            export_fp16(onnx_path, fp16_path)
            for old in sorted(glob.glob(f"{opts.output_dir}/snapshot_epoch_*.pth"), key=os.path.getmtime)[:-opts.keep_snapshots]:
                os.remove(old)
            print(f"Snapshot:  {snapshot_path}")
            print(f"ONNX:      {onnx_path}")
            print(f"ONNX fp16: {fp16_path}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = Generator(
    image_size=image_size,
    z_dim=z_dim,
    w_dim=w_dim,
    c_dim=c_dim,
    channel_base=channel_base,
    channel_max=channel_max,
    mapping_layers=mapping_layers,
    enc_scale=encoder_scale,
    enc_top_conv=encoder_top_conv,
    skip=skip_mode,
).to(device)

discriminator = Discriminator(
    image_size=image_size,
    channel_base=d_channel_base,
    channel_max=channel_max,
).to(device)

print(f"Generator params:     {sum(p.numel() for p in generator.parameters()):,} (channels {generator.chans})")
print(f"Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

opts = SimpleNamespace(
    output_dir=output_dir,
    log_file_path=log_file_path,
    image_size=image_size,
    z_dim=z_dim,
    epochs=epochs,
    lr=lr,
    lambda_l1=lambda_l1,
    lambda_vgg=lambda_vgg,
    r1_every=r1_every,
    r1_gamma=r1_gamma,
    ema_kimg=ema_kimg,
    seed=seed,
    sample_interval=sample_interval,
    snapshot_interval=snapshot_interval,
    keep_snapshots=keep_snapshots,
    compile_models=compile_models,
    fixed_cond=fixed_cond,
    fixed_target=fixed_target,
    val_cond=val_cond,
    val_target=val_target,
    restart=False,
)

train(generator, discriminator, dataloader, opts)